Link budget analysis, the accounting of gains and losses experienced by a signal as it travels from transmitter to receiver.

Conventions used: powers are supplied in **W**, antenna gains and line losses are supplied as **linear ratios** and converted to dB inside each function, frequency is in **MHz**, distance in **km**, bandwidth in **Hz**, data rate in **bps**. Environmental losses (atmosphere, rain, etc.) are supplied directly in **dB**.

In [1]:
import numpy as np

C = 299_792_458        # speed of light, m/s
K = 1.380649e-23       # boltzmann constant, J/K
T0 = 290               # reference temperature, K

In [2]:
def wavelength(frequency: float) -> float:
    """this function calculates the wavelength (m) from the frequency (MHz)"""
    return float(C / (frequency * 1e6))

In [3]:
def fspl(frequency: float, distance: float) -> float:
    """this function calculates the free space path loss (dB), frequency in MHz and distance in km"""
    fspl = 32.45 + 20 * np.log10(frequency) + 20 * np.log10(distance)
    return float(fspl)

In [40]:
fspl(2, 500000)

152.45

In [30]:
def fspl_db(distance_m, frequency_hz):
    """Free-space path loss in dB."""
    return 20 * np.log10(4 * np.pi * distance_m / wavelength(frequency_hz))

In [44]:
fspl_db(500000, 2000)

152.44778322188336

In [4]:
def eirp(transmit_power: float, transmit_gain: float, transmit_loss: float) -> float:
    """this function calculates the effective isotropic radiated power (EIRP) in dBW"""
    eirp = 10 * np.log10(transmit_power) + 10 * np.log10(transmit_gain) - 10 * np.log10(transmit_loss)
    return float(eirp)

In [27]:
eirp(12, 2, 1.32)

12.596373105057562

In [ ]:
def other_losses(atmospheric: float = 0, rain: float = 0, polarization: float = 0,
                 pointing: float = 0, obstruction: float = 0, other: float = 0) -> float:
    
    """this function adds up the remaining path losses (all in dB)"""

    return float(atmospheric + rain + polarization + pointing + obstruction + other)

In [45]:
other_losses(atmospheric=0.5, rain=0, polarization=0, pointing=0.5, obstruction=0, other=0)

1.0

In [ ]:
def rain_attenuation(rain_rate: float, k: float, alpha: float, path_length: float) -> float:
    """this function calculates a simplified ITU-R P.618 rain attenuation (dB).
    rain_rate in mm/h, k and alpha are the ITU-R P.838 coefficients, path_length in km"""
    
    specific_attenuation = k * rain_rate ** alpha
    l0 = 35 * np.exp(-0.015 * rain_rate)
    effective_length = path_length / (1 + path_length / l0)
    return float(specific_attenuation * effective_length)

In [ ]:
def received_power_friis_equation(transmit_power: float, transmit_gain: float, transmit_loss: float,
                                  receive_gain: float, receive_loss: float,
                                  path_loss: float, extra_losses: float = 0) -> float:
    
    """this function calculates the received power (dBW) using the Friis transmission equation.
    path_loss is the free space path loss (dB), extra_losses is the sum of the other path losses (dB)"""
    
    receiver_power = (eirp(transmit_power, transmit_gain, transmit_loss) - path_loss - extra_losses
                      + 10 * np.log10(receive_gain) - 10 * np.log10(receive_loss))
    return float(receiver_power)

In [46]:
received_power_friis_equation(12, 2, 1.32, 100, 1.58, 1)

29.609802235513335

In [8]:
def effective_aperture(receiver_gain: float, wavelength: float) -> float:
    """this function calculates the effective aperture of the receiver (m^2), gain is a linear ratio"""
    effective_aperture = (receiver_gain * (wavelength ** 2)) / (4 * np.pi)
    return float(effective_aperture)

In [9]:
def noise_figure(noise_factor: float) -> float:
    """this function converts a noise factor (linear) to a noise figure (dB)"""
    return float(10 * np.log10(noise_factor))

In [10]:
def system_noise_temperature(noise_figure: float, antenna_temperature: float, receive_loss: float) -> float:
    """this function calculates the system noise temperature (K) referred to the receiver input.
    noise_figure in dB, receive_loss is the linear receive line loss"""
    receiver_temperature = T0 * (10 ** (noise_figure / 10) - 1)
    return float(antenna_temperature / receive_loss + T0 * (1 - 1 / receive_loss) + receiver_temperature)

In [11]:
def figure_of_merit(receive_gain: float, receive_loss: float, system_temperature: float) -> float:
    """this function calculates the receiver figure of merit G/T (dB/K)"""
    return float(10 * np.log10(receive_gain) - 10 * np.log10(receive_loss) - 10 * np.log10(system_temperature))

In [12]:
def noise_power_density(system_temperature: float) -> float:
    """this function calculates the noise power density N0 (dBW/Hz)"""
    return float(10 * np.log10(K * system_temperature))

In [13]:
def noise_power(system_temperature: float, bandwidth: float) -> float:
    """this function calculates the noise power k T B (dBW)"""
    return float(10 * np.log10(K * system_temperature * bandwidth))

In [14]:
def carrier_to_noise_density(received_power: float, system_temperature: float) -> float:
    """this function calculates the carrier to noise density ratio C/N0 (dB-Hz)"""
    return float(received_power - noise_power_density(system_temperature))

In [15]:
def carrier_noise_ratio(carrier_power: float, noise_power: float) -> float:
    """this function calculates the carrier to noise ratio (dB), both powers in dBW"""
    cnr = carrier_power - noise_power
    return float(cnr)

In [16]:
def energy_per_bit_noise_density(cn0: float, data_rate: float, implementation_loss: float = 0) -> float:
    """this function calculates Eb/N0 (dB) from C/N0 (dB-Hz), data rate (bps) and implementation loss (dB)"""
    return float(cn0 - 10 * np.log10(data_rate) - implementation_loss)

In [17]:
def required_carrier_noise_ratio(required_ebn0: float, data_rate: float, bandwidth: float) -> float:
    """this function converts a required Eb/N0 (dB) to the required C/N (dB)"""
    return float(required_ebn0 + 10 * np.log10(data_rate) - 10 * np.log10(bandwidth))

In [18]:
def receiver_sensitivity(required_cnr: float, noise_power: float) -> float:
    """this function calculates the receiver sensitivity (dBW), the minimum received power that closes the link"""
    return float(required_cnr + noise_power)

In [19]:
def link_margin(received_power: float, required_power: float) -> float:
    """this function calculates the link margin (dB), both powers in dBW"""
    link_margin = received_power - required_power
    return float(link_margin)

Putting it all together with an example S-band downlink

In [20]:
# inputs
transmit_power = 5            # W
transmit_gain = 4             # linear (about 6 dBi)
transmit_loss = 1.26          # linear (about 1 dB)
receive_gain = 3162           # linear (about 35 dBi)
receive_loss = 1.12           # linear (about 0.5 dB)
frequency = 2200              # MHz
distance = 1500               # km
bandwidth = 2e6               # Hz
data_rate = 1e6               # bps
antenna_temperature = 150     # K
nf = 1.5                      # dB
implementation_loss = 1.0     # dB
required_ebn0 = 9.6           # dB

rain = rain_attenuation(rain_rate=20, k=0.00154, alpha=0.963, path_length=5)
extra = other_losses(atmospheric=0.3, rain=rain, polarization=0.5, pointing=1.0, other=0.5)

lam = wavelength(frequency)
path_loss = fspl(frequency, distance)
radiated = eirp(transmit_power, transmit_gain, transmit_loss)
pr = received_power_friis_equation(transmit_power, transmit_gain, transmit_loss, receive_gain, receive_loss, path_loss, extra)
aperture = effective_aperture(receive_gain, lam)

tsys = system_noise_temperature(nf, antenna_temperature, receive_loss)
gt = figure_of_merit(receive_gain, receive_loss, tsys)
n0 = noise_power_density(tsys)
n = noise_power(tsys, bandwidth)
cn0 = carrier_to_noise_density(pr, tsys)
cnr = carrier_noise_ratio(pr, n)
ebn0 = energy_per_bit_noise_density(cn0, data_rate, implementation_loss)

req_cnr = required_carrier_noise_ratio(required_ebn0, data_rate, bandwidth)
sensitivity = receiver_sensitivity(req_cnr, n)

print(f"wavelength            : {lam:.4f} m")
print(f"EIRP                  : {radiated:.2f} dBW")
print(f"FSPL                  : {path_loss:.2f} dB")
print(f"other losses          : {extra:.2f} dB")
print(f"received power        : {pr:.2f} dBW")
print(f"effective aperture    : {aperture:.2f} m^2")
print(f"system temperature    : {tsys:.2f} K")
print(f"G/T                   : {gt:.2f} dB/K")
print(f"N0                    : {n0:.2f} dBW/Hz")
print(f"noise power (kTB)     : {n:.2f} dBW")
print(f"C/N0                  : {cn0:.2f} dB-Hz")
print(f"C/N                   : {cnr:.2f} dB")
print(f"Eb/N0                 : {ebn0:.2f} dB")
print(f"required C/N          : {req_cnr:.2f} dB")
print(f"receiver sensitivity  : {sensitivity:.2f} dBW")
print()
print(f"Eb/N0 margin          : {ebn0 - required_ebn0:.2f} dB")
print(f"C/N margin            : {cnr - req_cnr:.2f} dB")
print(f"sensitivity margin    : {link_margin(pr, sensitivity):.2f} dB")

wavelength            : 0.1363 m
EIRP                  : 12.01 dBW
FSPL                  : 162.82 dB
other losses          : 2.42 dB
received power        : -118.72 dBW
effective aperture    : 4.67 m^2
system temperature    : 284.64 K
G/T                   : 9.96 dB/K
N0                    : -204.06 dBW/Hz
noise power (kTB)     : -141.05 dBW
C/N0                  : 85.33 dB-Hz
C/N                   : 22.32 dB
Eb/N0                 : 24.33 dB
required C/N          : 6.59 dB
receiver sensitivity  : -134.46 dBW

Eb/N0 margin          : 14.73 dB
C/N margin            : 15.73 dB
sensitivity margin    : 15.73 dB


Sanity check: with no extra losses the dB calculation must match the Friis equation in linear form, Pr = Pt Gt Gr (lambda / 4 pi d)^2

In [21]:
pr_db = received_power_friis_equation(transmit_power, transmit_gain, 1, receive_gain, 1, path_loss)
pr_linear = transmit_power * transmit_gain * receive_gain * (lam / (4 * np.pi * distance * 1e3)) ** 2
print(pr_db, 10 * np.log10(pr_linear))
assert np.isclose(pr_db, 10 * np.log10(pr_linear), atol=0.01)

-114.81036018495604 -114.8081434068394
